# DataFrames Basics

## Why DataFrames over RDDs?

DataFrames were introduced in Spark 1.3 and are now the recommended API for structured data. They offer three major advantages over raw RDDs:

| Feature | RDD | DataFrame |
|---------|-----|-----------|
| **Schema** | None — just typed collections | Named, typed columns with an explicit schema |
| **Optimizer** | None — runs exactly what you write | **Catalyst optimizer** rewrites your query for efficiency |
| **SQL** | Not available | Full SQL support via `spark.sql()` |
| **Performance** | JVM overhead for Python lambdas | Vectorized execution in the JVM, often 10-100x faster |
| **Readability** | Functional chains | Declarative, table-like operations |

Under the hood, a DataFrame *is* an RDD of `Row` objects, but Spark can optimize it because it knows the schema ahead of time.

In this notebook we will:
- Create a DataFrame from a Python list with an explicit schema
- Inspect structure with `show()`, `printSchema()`, and `dtypes`
- Apply selections, filters, and column transformations
- Use `pyspark.sql.functions` for expressive column expressions
- Generate summary statistics with `describe()`

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("03-DataFrames-Basics")
    .getOrCreate()
)

print("SparkSession ready. Version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 11:49:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready. Version: 3.5.3


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define an explicit schema using StructType.
# This is best practice — it avoids the cost of schema inference
# and makes the expected shape of the data self-documenting.
schema = StructType([
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=True),
    StructField("age",        IntegerType(), nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
])

# Raw data as a list of tuples — order must match the schema definition above
data = [
    ("Alice",   "Engineering", 30, 95000.0),
    ("Bob",     "Marketing",   45, 72000.0),
    ("Carol",   "Engineering", 27, 88000.0),
    ("David",   "HR",          33, 65000.0),
    ("Eve",     "Engineering", 25, 91000.0),
    ("Frank",   "Marketing",   38, 78000.0),
    ("Grace",   "HR",          29, 67000.0),
    ("Hector",  "Engineering", 42, 105000.0),
]

# createDataFrame() distributes the data and binds the schema
df = spark.createDataFrame(data, schema=schema)

print("DataFrame created with", df.count(), "rows.")

26/06/24 11:49:51 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (172.18.0.4 executor 1): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 8) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.

Py4JJavaError: An error occurred while calling o39.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 0.0 failed 4 times, most recent failure: Lost task 2.3 in stage 0.0 (TID 15) (172.18.0.4 executor 1): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 8) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scal

In [ ]:
# .show() — print a tabular preview of the data (default 20 rows)
# truncate=False ensures long values are not cut off
print("── df.show() ──")
df.show(truncate=False)

# .printSchema() — display the tree-structured schema with types and nullability
print("── df.printSchema() ──")
df.printSchema()

# .dtypes — Python list of (column_name, type_string) tuples
print("── df.dtypes ──")
for col_name, col_type in df.dtypes:
    print(f"  {col_name:<12} -> {col_type}")

# .count() — total number of rows (triggers a job)
print("\nTotal rows:", df.count())

In [ ]:
# ── Selecting columns ─────────────────────────────────────────────────────
# .select() returns a new DataFrame with only the specified columns
print("── select(name, age) ──")
df.select("name", "age").show()

# ── Filtering rows ────────────────────────────────────────────────────────
# .filter() (alias: .where()) keeps rows that satisfy a boolean condition
print("── filter(age > 30) ──")
df.filter(df.age > 30).show()

# ── Adding / replacing columns ────────────────────────────────────────────
# .withColumn() returns a new DataFrame with a column added or replaced.
# It does NOT mutate the original DataFrame (DataFrames are immutable).
print("── withColumn: age_next_year ──")
df.withColumn("age_next_year", df.age + 1).select("name", "age", "age_next_year").show()

## Column Expressions — `col()`, `lit()`, and `F.when()`

Instead of referencing columns directly as `df.column_name`, it is often cleaner to use helper functions from `pyspark.sql.functions`:

| Function | Purpose | Example |
|----------|---------|--------|
| `col("name")` | Reference a column by name string | `col("age") > 30` |
| `lit(value)` | Create a column with a constant literal value | `lit(2024)` |
| `F.when(cond, val).otherwise(val)` | Conditional expression (like SQL CASE WHEN) | `F.when(col("age") >= 35, "senior").otherwise("junior")` |

Using `col()` and `F.*` functions is preferred over `df.column_name` notation because:
- Column references are decoupled from a specific DataFrame object
- They compose cleanly inside complex expressions
- They work inside `.select()` and `.withColumn()` without ambiguity

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit

# col() — reference a column by name; works independently of the DataFrame variable
# lit() — wrap a Python scalar into a Spark Column of constant value
# F.when().otherwise() — vectorised conditional, equivalent to SQL CASE WHEN

df_enriched = (
    df
    # Add a constant column (useful for tagging rows with a batch ID, version, etc.)
    .withColumn("cohort_year", lit(2024))

    # Conditional: classify employees as junior or senior based on age
    .withColumn(
        "seniority",
        F.when(col("age") >= 35, "senior").otherwise("junior")
    )

    # Compute salary in thousands (rounded to 1 decimal) using col() expression
    .withColumn(
        "salary_k",
        F.round(col("salary") / lit(1000), 1)
    )
)

df_enriched.select("name", "age", "seniority", "salary_k", "cohort_year").show()

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────
# .describe() computes count, mean, stddev, min, max for numeric
# and string columns. Returns a new DataFrame — call .show() to display it.

print("── df.describe() ──")
df.describe("age", "salary").show()

# For more precise statistics, use .summary() which also includes percentiles
print("── df.summary() ──")
df.select("age", "salary").summary().show()

In [3]:
# Release cluster resources
spark.stop()
print("SparkSession stopped. All done!")

SparkSession stopped. All done!
